# Notebook 9 — Amazon Reviews 2023 (text + photo): text processing, fairness and fusion, 3 seeds

**What this notebook answers** (all on the same 12,000 reviews and the **same product split as Notebook 1**, checked by fingerprint):
1. **Text:** does conventional cleaning hurt (run 1: −8.2 pp)? Does ECCM help? Does Group-DRO help the worst category, and at what cost?
2. **Photo:** does the customer photo add anything, is any gain due to the *real* review–photo pairing, and does our more complex
   fusion beat the paper's RMSR (Eq. 10–14) or a simple concatenation?
3. **Missing photos:** how much does each fusion model lose when photos are hidden (identical hidden-photo masks for every model)?

**Our RMSR adaptation is NOT the published EMO-FAR model.**
| Element | Published EMO-FAR | This notebook |
|---|---|---|
| Text processing | ECCM as described in the paper | Our implementation of the described cues (Notebook 1): negation, negation scope, intensifier, contrast, informal cues, PMI domain anchoring |
| Encoders and training | The paper's own encoders and end-to-end training | RoBERTa-base fine-tuned on text (2 epochs, as Notebook 1); CLIP ViT-B/32 image encoder, **frozen**; fusion heads trained on the frozen features |
| Fusion | RMSR, Eq. 10–14 | Eq. 10–14 applied to the two frozen feature vectors (our adaptation, same code pattern as M2 in Notebooks 2–8) |
| Alignment loss Eq. 15 (λ1) | used | **not used** (Notebook 8: no benefit; harmful without text on MOSEI) |
| Latent (λ2) and fairness (λ3) losses | used | not used; **Group-DRO** is our fairness objective (only if validation selects it) |
| Data | the paper's datasets | Amazon Reviews 2023, Fashion + Beauty reviews with a customer photo, 12,000, balanced |
Numbers here are therefore **not** a reproduction of the paper's reported results.

**What this dataset is — and is not** (read before interpreting any number below)
- **Only reviews that have a customer photo** (Fashion and Beauty). Reviews with photos are a selected subset of all reviews; results
  may not hold for reviews without photos.
- **Sentiment labels come from star ratings** (1–2★ negative, 3★ neutral, 4–5★ positive), not from human sentiment annotation.
  They are noisy: the text or photo may not match the star rating.
- **Classes are balanced by construction** (2,000 per class per category), not the natural rating distribution.
- **Missing photos are simulated** by hiding photos with fixed masks. About 1 % of photos also failed to download (real missingness,
  treated as "no photo" for every model).
- **Fairness groups are product categories** (Fashion vs Beauty), not demographic groups. "Worst-category F1" and "category gap"
  measure performance differences between product categories only.

**Models**
| ID | What | Trained on | Seeds |
|---|---|---|---|
| R1 | conventional cleaning + RoBERTa | text | 3 |
| R2 | raw text + RoBERTa | text | 3 |
| R3 | ECCM + RoBERTa | text | 3 |
| R4 | ECCM + RoBERTa + Group-DRO | text | 3 |
| — | **text backbone for all heads below = R1–R4 chosen on validation only** (rule below; Group-DRO is not assumed) | | |
| T0 | text-only head (re-fit of the text classifier on the frozen text vector; the matched reference) | frozen text | 3 |
| IMG | image-only lightweight head (MLP on the mean CLIP token) | frozen photo | 3 |
| CAT | simple concatenation [text ; photo] → MLP | both | 3 |
| F1 | **paper's RMSR adaptation** (Eq. 10–14, 2 modalities, as M2) — main | both | 3 |
| F2 | Notebook 1's design: token-level cross-attention (text tokens → 50 photo tokens) + masked gate | both | 3 |
| F3 | lightweight pooled attention (B3-style: W_v·mean + mean) + masked gate | both | 3 |
| SHUF | F1 trained and tested with photos **shuffled across reviews** (breaks the real pairing) | both | 3 |
| F1-shuffled-test | the **trained F1 checkpoint**, evaluated with test photos shuffled **within each product category** among reviews that have a photo (every review keeps "has a photo", but gets another review's photo; 5 shuffles averaged). No training | — | 3 |
F1's parameter count includes Eq. 13's W_q and W_k, kept for fidelity to M2, although they have no effect (softmax over one key = 1).
All fusion heads use modality dropout 0.3 (as Notebook 1). Head seed s uses the text model of seed s (paired).
Text runs: 12 RoBERTa fine-tunings. Heads: 21 small trainings. The 12,000 photos are downloaded and encoded once and **cached**.

**Text-backbone rule (validation only, fixed in advance):** among R1–R4 take the highest mean validation macro-F1; every model within
0.5 pp of it is tied; among tied models take the highest mean validation worst-category macro-F1; if still within 0.5 pp, take the
simplest (R2 < R1 < R3 < R4).

**Primary tests** (seed + product bootstrap, 10,000 draws; Holm within each family; verdict rule as Notebook 8:
HELPS/HURTS = |effect| ≥ 1 pp and Holm p < .05; negligible = 95 % CI inside ±1 pp; otherwise no detectable effect)
| Family | Test | Measure |
|---|---|---|
| Text (4) | T1 R2 − R1 · T2 R3 − R2 · T3 R4 − R3 | test macro-F1 |
| | T4 R4 − R3 | test worst-category macro-F1 |
| Fusion (6) | P1 F1 − T0 (photo helps?) · P2 F1 − SHUF (real pairing?) · P3 CAT − T0 · P4 F2 − F1 · P5 F3 − F1 · P6 CAT − F1 | test macro-F1, all photos present |
Secondary (Holm within): photos hidden for 100 % of test reviews minus 0 % for each fusion model; and F1 with its real test photos
minus F1 with within-category shuffled test photos (evaluation-only pairing check, complementing the trained SHUF control). Descriptive: IMG vs chance,
worst-category macro-F1, category gap, ECE, macro-F1 on reviews with a negation word, inference latency, parameter count.

**Paired masks:** the hidden-photo masks (30/50/70 % × 5 mask seeds, and 100 %) are generated once, saved, and applied to every model.
Reviews whose photo could not be downloaded count as "no photo" for every model.

**Runtime: GPU (L4 or A100), High-RAM.** Estimated 1–1.5 h; the first text run and the first head are timed and a projection is
printed. Everything is saved to `emofar_results/amazon_v9/` as it finishes; after a disconnect, **Run all** continues.

In [1]:
import os, sys, time, json, gc
IN_COLAB = "google.colab" in sys.modules
TEST_MODE = os.environ.get("EMOFAR_TEST") == "1"
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
import torch, warnings
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()
warnings.filterwarnings("ignore", category=UserWarning)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU")
if IN_COLAB and DEVICE != "cuda":
    raise RuntimeError("Please switch to a GPU runtime (Runtime -> Change runtime type -> L4 GPU).")

Mounted at /content/drive
Device: cuda | NVIDIA L4


## 1. Settings (text settings identical to Notebook 1)

In [2]:
DATA_DIR = "/content/drive/MyDrive/datasets/Amazon-Reviews-2023/raw/review_categories"
OUT_DIR = "/content/drive/MyDrive/emofar_results/amazon_v9"
CATEGORIES = ["Amazon_Fashion", "All_Beauty"]
MODEL_NAME = "roberta-base"
CLIP_NAME = "openai/clip-vit-base-patch32"
N_PER_CLASS_PER_CAT = 2000
MAX_LEN, EPOCHS, BATCH_SIZE = 128, 2, 32
LR_ENCODER, LR_NEW = 2e-5, 1e-4
SEEDS = [42, 43, 44]
GDRO_ETA = 0.01
IMG_DROPOUT = 0.3
USE_IMAGES = True                       # Notebook 1's model code reads this name; image features are used by the heads below
D_HEAD, HEAD_EPOCHS, HEAD_PATIENCE, HEAD_LR, HEAD_BS = 256, 30, 5, 1e-3, 64
N_BOOT, MARGIN_PP, ALPHA = 10000, 1.0, 0.05
MISS_RATES, MASK_SEEDS = [0.3, 0.5, 0.7], [0, 1, 2, 3, 4]
EXPECTED_SPLIT_FP = "c3a53bf95269b4f6e8a490d0d85be4f9"   # Notebook 1's split (12,000 reviews), computed from the same files
if TEST_MODE:
    DATA_DIR, OUT_DIR = os.environ["EMOFAR_DATA"], os.environ["EMOFAR_OUT"]
    MODEL_NAME, MAX_LEN, BATCH_SIZE = "distilroberta-base", 64, 16
    N_PER_CLASS_PER_CAT = int(os.environ.get("EMOFAR_N", 40)); EPOCHS = int(os.environ.get("EMOFAR_EPOCHS", 1))
    HEAD_EPOCHS, N_BOOT = 4, 500
for sub in ["text", "heads", "clip_chunks"]:
    os.makedirs(f"{OUT_DIR}/{sub}", exist_ok=True)
LABELS = ["negative", "neutral", "positive"]
CONFIG = dict(MODEL_NAME=MODEL_NAME, N_PER_CLASS_PER_CAT=N_PER_CLASS_PER_CAT, MAX_LEN=MAX_LEN, EPOCHS=EPOCHS, BATCH_SIZE=BATCH_SIZE,
              LR_ENCODER=LR_ENCODER, LR_NEW=LR_NEW, GDRO_ETA=GDRO_ETA, IMG_DROPOUT=IMG_DROPOUT, D_HEAD=D_HEAD, HEAD_EPOCHS=HEAD_EPOCHS,
              HEAD_PATIENCE=HEAD_PATIENCE, HEAD_LR=HEAD_LR, HEAD_BS=HEAD_BS, CLIP_NAME=CLIP_NAME)
import transformers, platform
print({"torch": torch.__version__, "transformers": transformers.__version__, "python": platform.python_version()})
json.dump({"config": CONFIG, "seeds": SEEDS, "torch": torch.__version__, "transformers": transformers.__version__,
           "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"},
          open(f"{OUT_DIR}/run_info_{time.strftime('%Y%m%d_%H%M%S', time.gmtime())}.json", "w"), indent=2)

{'torch': '2.14.1+cu130', 'transformers': '5.22.1', 'python': '3.13.16'}


## 2. Data: Notebook 1's loading, de-duplication, balancing and product split (code unchanged)

In [3]:
import json, re, random, hashlib
import numpy as np, pandas as pd

def star_to_label(stars):
    return 0 if stars <= 2 else (1 if stars == 3 else 2)

rows = []
for cat in CATEGORIES:
    path = f"{DATA_DIR}/{cat}.jsonl"
    n_cat = 0
    with open(path, encoding="utf8") as f:
        for line in f:
            if '"images": []' in line:          # fast skip: review has no photo
                continue
            r = json.loads(line)
            if not r.get("images"):
                continue
            text = f"{r.get('title', '').strip()}. {r.get('text', '').strip()}".strip(". ").strip()
            if len(text.split()) < 3:
                continue
            rows.append({"text": text, "label": star_to_label(int(r["rating"])), "stars": int(r["rating"]),
                         "category": cat, "parent_asin": r["parent_asin"],
                         "image_url": r["images"][0].get("small_image_url", "")})
            n_cat += 1
    print(f"{cat}: {n_cat:,} reviews with photos")

df_all = pd.DataFrame(rows)
print("Total:", len(df_all))
print(pd.crosstab(df_all.category, df_all.label.map(dict(enumerate(LABELS)))))

def norm_key(t):
    return hashlib.md5(re.sub(r"\s+", " ", t.lower()).strip().encode()).hexdigest()

df_all["key"] = df_all.text.map(norm_key)
before = len(df_all)
df_all = df_all.drop_duplicates("key").reset_index(drop=True)
print(f"Removed {before - len(df_all):,} duplicate reviews")

parts = []
for cat in CATEGORIES:
    for lab in range(3):
        sub = df_all[(df_all.category == cat) & (df_all.label == lab)]
        n = min(N_PER_CLASS_PER_CAT, len(sub))
        parts.append(sub.sample(n=n, random_state=42))
df = pd.concat(parts).reset_index(drop=True)

rng = np.random.RandomState(42)
products = np.array(sorted(df.parent_asin.unique()), dtype=object)
rng.shuffle(products)
n_p = len(products)
split_of = {p: "train" for p in products[: int(0.70 * n_p)]}
split_of.update({p: "valid" for p in products[int(0.70 * n_p): int(0.85 * n_p)]})
split_of.update({p: "test" for p in products[int(0.85 * n_p):]})
df["split"] = df.parent_asin.map(split_of)

print(pd.crosstab([df.split, df.category], df.label.map(dict(enumerate(LABELS)))))
del df_all, rows                                  # free memory: the full table is no longer needed
import gc; gc.collect()
assert not (set(df[df.split == "train"].parent_asin) & set(df[df.split == "test"].parent_asin)), "product leak!"
print("Check passed: no product appears in both train and test.")

import hashlib as _hl
SPLIT_FP = _hl.md5("|".join(f"{k}:{s}" for k, s in sorted(zip(df.key, df.split))).encode()).hexdigest()
print("Split fingerprint:", SPLIT_FP)
if not TEST_MODE:
    assert SPLIT_FP == EXPECTED_SPLIT_FP, "The split differs from Notebook 1's - stop and report (data files or code changed)."
    print("Check passed: identical reviews and identical product split as Notebook 1.")
split_file = f"{OUT_DIR}/split.csv"
if os.path.exists(split_file):
    old = pd.read_csv(split_file)
    assert list(old.key) == list(df.key) and list(old.split) == list(df.split), "Saved split differs from the current one"
else:
    df[["key", "split", "parent_asin", "category", "label", "stars", "image_url"]].to_csv(split_file, index=False)

Amazon_Fashion: 158,765 reviews with photos
All_Beauty: 59,357 reviews with photos
Total: 218122
label           negative  neutral  positive
category                                   
All_Beauty         11335     3912     44110
Amazon_Fashion     33224    12017    113524
Removed 2,869 duplicate reviews
label                 negative  neutral  positive
split category                                   
test  All_Beauty           303      311       343
      Amazon_Fashion       285      306       285
train All_Beauty          1382     1385      1363
      Amazon_Fashion      1422     1402      1413
valid All_Beauty           315      304       294
      Amazon_Fashion       293      292       302
Check passed: no product appears in both train and test.
Split fingerprint: c3a53bf95269b4f6e8a490d0d85be4f9
Check passed: identical reviews and identical product split as Notebook 1.


## 3. ECCM, conventional cleaning, PMI domain anchoring, encoding (Notebook 1, unchanged)

In [4]:
NEGATIONS = {"not", "no", "never", "nothing", "nobody", "none", "neither", "nor", "cannot", "without",
             "hardly", "barely", "scarcely", "nowhere", "cant", "dont", "doesnt", "didnt", "isnt", "wasnt",
             "wont", "wouldnt", "shouldnt", "couldnt", "arent", "werent", "havent", "hasnt", "hadnt", "aint"}
INTENSIFIERS = {"very", "really", "so", "too", "extremely", "super", "totally", "absolutely", "highly",
                "completely", "incredibly", "quite", "truly", "most", "much", "way", "seriously",
                "terribly", "awfully", "insanely", "utterly", "especially", "particularly", "even"}
CONTRAST = {"but", "however", "although", "though", "yet", "except", "whereas", "nevertheless", "unfortunately"}
SCOPE_BREAK = set(".,!?;:") | CONTRAST
EMOJI_RE = re.compile("[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F000-\U0001F2FF]")
WORD_RE = re.compile(r"\w+(?:'\w+)?|[^\w\s]")
FLAG_NAMES = ["NEG", "SCOPE", "INT", "CON", "INF"]

def eccm_words(text):
    """Split text into words and compute the 5 cue flags for each word."""
    words, flags = [], []
    for w in WORD_RE.findall(text):
        informal = bool(EMOJI_RE.search(w)) or bool(re.search(r"(\w)\1{2,}", w)) \
                   or (w.isupper() and len(w) >= 3 and w.isalpha()) or w == "!"
        w_norm = re.sub(r"(\w)\1{2,}", r"\1\1", w)                    # sooooo -> soo
        low = w_norm.lower().replace("'", "")
        f = [0, 0, 0, 0, 0]
        f[0] = int(low in NEGATIONS or w_norm.lower().endswith("n't"))
        f[2] = int(low in INTENSIFIERS)
        f[3] = int(low in CONTRAST)
        f[4] = int(informal)
        words.append(w_norm); flags.append(f)
    scope_left = 0                                                     # negation scope: next <=3 words
    for i, w in enumerate(words):
        if flags[i][0]:
            scope_left = 3; continue
        if w.lower() in SCOPE_BREAK:
            scope_left = 0; continue
        if scope_left > 0:
            flags[i][1] = 1; scope_left -= 1
    return words, flags

w, f = eccm_words("Not very good, but the colour is sooooo NICE 😍!!")
for word, fl in zip(w, f):
    print(f"{word:8s}", [n for n, v in zip(FLAG_NAMES, fl) if v])

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
print("Is 'not' a stop-word?", "not" in ENGLISH_STOP_WORDS, "| 'never'?", "never" in ENGLISH_STOP_WORDS)

def conventional_clean(text):
    t = re.sub(r"https?://\S+", " ", text.lower())
    t = re.sub(r"[^a-z\s]", " ", t)                  # drops emojis, digits, punctuation, apostrophes
    return " ".join(w for w in t.split() if w not in ENGLISH_STOP_WORDS)

print(conventional_clean("Not very good, but the colour is sooooo NICE 😍!!"))

from collections import Counter, defaultdict

df["eccm"] = df.text.map(eccm_words)
train_df = df[df.split == "train"]
word_cat, word_all, cat_total = defaultdict(Counter), Counter(), Counter()
for (words, _), cat in zip(train_df.eccm, train_df.category):
    for w_ in set(x.lower() for x in words):
        word_cat[cat][w_] += 1; word_all[w_] += 1
    cat_total[cat] += 1
N_TRAIN = len(train_df)

ANCHOR_SKIP = set(ENGLISH_STOP_WORDS) - NEGATIONS     # filler words are not domain words

def pmi(word, cat, min_count=10):
    c_all = word_all.get(word, 0)
    if c_all < min_count or not word.isalpha() or len(word) < 3 or word in ANCHOR_SKIP:
        return 0.0
    p_w_given_c = word_cat[cat].get(word, 0) / max(cat_total[cat], 1)
    p_w = c_all / N_TRAIN
    if p_w_given_c == 0:
        return 0.0
    return float(np.clip(np.log(p_w_given_c / p_w) / 3.0, 0.0, 1.0))

for cat in CATEGORIES:
    top = sorted(((pmi(w_, cat), w_) for w_ in word_cat[cat]), reverse=True)[:10]
    print(cat, "→", [w_ for _, w_ in top])

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, add_prefix_space=True)

def encode(frame, mode):
    """mode: 'conventional' | 'raw' | 'eccm'. Returns a dict of tensors."""
    ids, masks, flag_t, pmi_t = [], [], [], []
    for text, (words, flags), cat in zip(frame.text, frame.eccm, frame.category):
        if mode == "conventional":
            words = conventional_clean(text).split() or ["empty"]
            flags = [[0] * 5 for _ in words]
        elif mode == "raw":
            words = WORD_RE.findall(text); flags = [[0] * 5 for _ in words]
        enc = tokenizer(words, is_split_into_words=True, truncation=True, max_length=MAX_LEN,
                        padding="max_length")
        wid = enc.word_ids()
        fl = [[0] * 5 if i is None else flags[i] for i in wid]
        pm = [0.0 if (i is None or mode != "eccm") else pmi(words[i].lower(), cat) for i in wid]
        ids.append(enc["input_ids"]); masks.append(enc["attention_mask"]); flag_t.append(fl); pmi_t.append(pm)
    return {"input_ids": torch.tensor(ids), "attention_mask": torch.tensor(masks),
            "flags": torch.tensor(flag_t, dtype=torch.float), "pmi": torch.tensor(pmi_t, dtype=torch.float),
            "label": torch.tensor(frame.label.values), "group": torch.tensor([CATEGORIES.index(c) for c in frame.category]),
            "row": torch.tensor(frame.index.values)}

splits = {s: df[df.split == s] for s in ["train", "valid", "test"]}
encoded = {mode: {s: encode(fr, mode) for s, fr in splits.items()} for mode in ["conventional", "raw", "eccm"]}
print({s: len(fr) for s, fr in splits.items()})

Not      ['NEG']
very     ['SCOPE', 'INT']
good     ['SCOPE']
,        []
but      ['CON']
the      []
colour   []
is       []
soo      ['INF']
NICE     ['INF']
😍        ['INF']
!        ['INF']
!        ['INF']
Is 'not' a stop-word? True | 'never'? True
good colour sooooo nice
Amazon_Fashion → ['wallet', 'shorts', 'yoga', 'underwear', 'tummy', 'tights', 'swimsuit', 'sterling', 'sports', 'soles']
All_Beauty → ['waves', 'watery', 'tattoos', 'tangling', 'skincare', 'sheds', 'scents', 'puffiness', 'pores', 'pony']


{'train': 8367, 'valid': 1800, 'test': 1833}


## 4. Customer photos → CLIP features (downloaded and encoded once, cached in chunks; resumes after a disconnect)

In [5]:
import requests, io
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
from transformers import CLIPVisionModel, CLIPImageProcessor

def fetch(url):
    try:
        r = requests.get(url, timeout=10); r.raise_for_status()
        return Image.open(io.BytesIO(r.content)).convert("RGB")
    except Exception:
        return None

CHUNK = 1024
urls = df.image_url.tolist(); n_chunks = (len(urls) + CHUNK - 1) // CHUNK
todo = [c for c in range(n_chunks) if not os.path.exists(f"{OUT_DIR}/clip_chunks/chunk_{c:03d}.npz")]
if todo:
    clip_proc = CLIPImageProcessor.from_pretrained(CLIP_NAME)
    clip = CLIPVisionModel.from_pretrained(CLIP_NAME).to(DEVICE).eval()
    t0 = time.time()
    with ThreadPoolExecutor(32) as pool, torch.no_grad():
        for k, c in enumerate(todo, 1):
            lo, hi = c * CHUNK, min((c + 1) * CHUNK, len(urls))
            images = list(pool.map(fetch, urls[lo:hi]))
            feats = np.zeros((hi - lo, 50, clip.config.hidden_size), dtype=np.float16); has = np.zeros(hi - lo, dtype=bool)
            ok = [i for i, im in enumerate(images) if im is not None]
            for b in range(0, len(ok), 64):
                sel = ok[b:b + 64]
                px = clip_proc(images=[images[i] for i in sel], return_tensors="pt")["pixel_values"].to(DEVICE)
                feats[sel] = clip(pixel_values=px).last_hidden_state.float().cpu().numpy().astype(np.float16); has[sel] = True
            tmp = f"{OUT_DIR}/clip_chunks/chunk_{c:03d}.tmp.npz"; np.savez(tmp, feats=feats, has=has)
            os.replace(tmp, f"{OUT_DIR}/clip_chunks/chunk_{c:03d}.npz")
            el = time.time() - t0
            print(f"  photo chunk {c + 1}/{n_chunks}: {has.sum()}/{hi - lo} photos | {el:.0f}s elapsed, about {el / k * (len(todo) - k):.0f}s left", flush=True)
    del clip; torch.cuda.empty_cache() if DEVICE == "cuda" else None
chunks = [np.load(f"{OUT_DIR}/clip_chunks/chunk_{c:03d}.npz") for c in range(n_chunks)]
IMG_FEATS = np.concatenate([z["feats"] for z in chunks]); HAS_IMG = np.concatenate([z["has"] for z in chunks]); del chunks
print(f"Photo features: {IMG_FEATS.shape}, photos available for {HAS_IMG.sum():,} / {len(HAS_IMG):,} reviews")

  photo chunk 1/12: 1014/1024 photos | 13s elapsed, about 142s left
  photo chunk 2/12: 1021/1024 photos | 25s elapsed, about 125s left
  photo chunk 3/12: 1015/1024 photos | 36s elapsed, about 107s left
  photo chunk 4/12: 1021/1024 photos | 47s elapsed, about 93s left
  photo chunk 5/12: 1018/1024 photos | 57s elapsed, about 80s left
  photo chunk 6/12: 1019/1024 photos | 69s elapsed, about 69s left
  photo chunk 7/12: 1010/1024 photos | 84s elapsed, about 60s left
  photo chunk 8/12: 1015/1024 photos | 94s elapsed, about 47s left
  photo chunk 9/12: 1017/1024 photos | 104s elapsed, about 35s left
  photo chunk 10/12: 1016/1024 photos | 114s elapsed, about 23s left
  photo chunk 11/12: 1019/1024 photos | 125s elapsed, about 11s left
  photo chunk 12/12: 727/736 photos | 139s elapsed, about 0s left
Photo features: (12000, 50, 768), photos available for 11,912 / 12,000 reviews


## 5. Text models R1–R4 (Notebook 1's model and training code, unchanged), 3 seeds — saved as each finishes

In [6]:
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel

class EmoFarModel(nn.Module):
    def __init__(self, use_eccm=False, use_image=False, img_dim=768):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        H = self.encoder.config.hidden_size
        self.use_eccm, self.use_image = use_eccm, use_image
        if use_eccm:
            self.cue_emb = nn.Parameter(torch.randn(5, H) * 0.02)      # E_k, one vector per cue type
            self.alpha = nn.Parameter(torch.tensor([0.1, 0.1]))          # α_INT, α_INF
            self.beta = nn.Parameter(torch.tensor(0.1))                  # β (domain anchoring)
        if use_image:
            self.img_proj = nn.Sequential(nn.Linear(img_dim, H), nn.LayerNorm(H))
            self.cross_attn = nn.MultiheadAttention(H, num_heads=8, batch_first=True)
            self.gate = nn.Linear(2 * H, 2)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(H, 3)

    def forward(self, input_ids, attention_mask, flags, pmi, img=None, has_img=None, return_gate=False):
        emb = self.encoder.embeddings.word_embeddings(input_ids)        # (B, L, H)
        if self.use_eccm:
            s = 1 + self.alpha[0] * flags[..., 2] + self.alpha[1] * flags[..., 4]
            a = 1 + self.beta * pmi
            emb = emb * (a * s).unsqueeze(-1) + flags @ self.cue_emb
        tok = self.encoder(inputs_embeds=emb, attention_mask=attention_mask).last_hidden_state
        h_text = tok[:, 0]                                               # <s> token summary
        gate_w = None
        if self.use_image:
            v = self.img_proj(img)                                       # (B, 50, H)
            attended, _ = self.cross_attn(tok, v, v)                     # text tokens attend to image tokens
            m = attention_mask.unsqueeze(-1).float()
            h_img = (attended * m).sum(1) / m.sum(1).clamp(min=1)
            logits_g = self.gate(torch.cat([h_text, h_img], -1))
            logits_g = logits_g.masked_fill(torch.stack([torch.zeros_like(has_img), ~has_img], -1), float("-inf"))
            gate_w = logits_g.softmax(-1)                                # (B, 2): [w_text, w_image]
            h = gate_w[:, :1] * h_text + gate_w[:, 1:] * h_img
        else:
            h = h_text
        out = self.classifier(self.dropout(h))
        return (out, gate_w) if return_gate else out

import time
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from transformers import get_linear_schedule_with_warmup

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def batches(data, bs, shuffle, seed=0):
    n = len(data["label"])
    order = np.random.RandomState(seed).permutation(n) if shuffle else np.arange(n)
    for i in range(0, n, bs):
        yield {k: v[order[i:i + bs]] for k, v in data.items()}

def model_inputs(b, use_image, drop_image_p=0.0, rng=None):
    x = {k: b[k].to(DEVICE) for k in ["input_ids", "attention_mask", "flags", "pmi"]}
    if use_image:
        rows = b["row"].numpy()
        has = HAS_IMG[rows].copy()
        if drop_image_p > 0:
            has &= rng.rand(len(has)) >= drop_image_p                   # modality dropout / missing-image test
        img = torch.tensor(IMG_FEATS[rows], dtype=torch.float).to(DEVICE)
        img = img * torch.tensor(has, dtype=torch.float, device=DEVICE)[:, None, None]
        x.update(img=img, has_img=torch.tensor(has, device=DEVICE))
    return x

def ece_score(probs, y, n_bins=15):
    conf, pred = probs.max(1), probs.argmax(1)
    ece, edges = 0.0, np.linspace(0, 1, n_bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        sel = (conf > lo) & (conf <= hi)
        if sel.any():
            ece += sel.mean() * abs((pred[sel] == y[sel]).mean() - conf[sel].mean())
    return float(ece)

@torch.no_grad()
def predict(model, data, use_image, drop_image_p=0.0, seed=0):
    model.eval(); probs = []
    rng = np.random.RandomState(seed)
    for b in batches(data, 64, False):
        with torch.autocast(DEVICE, dtype=torch.float16, enabled=DEVICE == "cuda"):
            out = model(**model_inputs(b, use_image, drop_image_p, rng))
        probs.append(out.float().softmax(-1).cpu().numpy())
    return np.concatenate(probs)

def evaluate(probs, data):
    y, g, pred = data["label"].numpy(), data["group"].numpy(), probs.argmax(1)
    rows = data["row"].numpy()
    has_neg = np.array([any(f[0] for f in df.loc[r, "eccm"][1]) for r in rows])
    per_group = {CATEGORIES[k]: f1_score(y[g == k], pred[g == k], average="macro") for k in np.unique(g)}
    res = {"accuracy": accuracy_score(y, pred), "macro_f1": f1_score(y, pred, average="macro"),
           "worst_group_f1": min(per_group.values()), "fairness_gap": max(per_group.values()) - min(per_group.values()),
           "ece": ece_score(probs, y),
           "negation_f1": f1_score(y[has_neg], pred[has_neg], average="macro") if has_neg.any() else float("nan")}
    res.update({f"f1_{k}": v for k, v in per_group.items()})
    return res

def train_run(name, mode, use_eccm, loss_type, use_image, seed):
    set_seed(seed)
    data = encoded[mode]
    model = EmoFarModel(use_eccm=use_eccm, use_image=use_image).to(DEVICE)
    enc_params = list(model.encoder.parameters())
    enc_ids = {id(p) for p in enc_params}
    new_params = [p for p in model.parameters() if id(p) not in enc_ids]
    opt = torch.optim.AdamW([{"params": enc_params, "lr": LR_ENCODER}, {"params": new_params, "lr": LR_NEW}], weight_decay=0.01)
    steps = EPOCHS * int(np.ceil(len(data["train"]["label"]) / BATCH_SIZE))
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")
    q = torch.ones(len(CATEGORIES), device=DEVICE) / len(CATEGORIES)    # Group-DRO weights
    rng = np.random.RandomState(seed)
    best_f1, best_state, t0 = -1, None, time.time()
    for ep in range(EPOCHS):
        model.train()
        for b in batches(data["train"], BATCH_SIZE, True, seed + ep):
            y, g = b["label"].to(DEVICE), b["group"].to(DEVICE)
            with torch.autocast(DEVICE, dtype=torch.float16, enabled=DEVICE == "cuda"):
                logits = model(**model_inputs(b, use_image, IMG_DROPOUT if use_image else 0.0, rng))
            per_sample = F.cross_entropy(logits.float(), y, reduction="none")
            if loss_type == "gdro":
                group_loss = torch.stack([per_sample[g == k].mean() if (g == k).any() else per_sample.new_zeros(())
                                          for k in range(len(CATEGORIES))])
                q = q * torch.exp(GDRO_ETA * group_loss.detach()); q = q / q.sum()
                loss = (q * group_loss).sum()
            else:
                loss = per_sample.mean()
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
        val = evaluate(predict(model, data["valid"], use_image), data["valid"])
        print(f"  [{name}] epoch {ep + 1}/{EPOCHS}  valid macro-F1 = {val['macro_f1']:.4f}  ({time.time() - t0:.0f}s)")
        if val["macro_f1"] > best_f1:
            best_f1 = val["macro_f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model

TEXT_RUNS = {"R1": ("conventional", False, "ce"), "R2": ("raw", False, "ce"), "R3": ("eccm", True, "ce"), "R4": ("eccm", True, "gdro")}
SIMPLICITY = ["R2", "R1", "R3", "R4"]

def text_states(model, data, want_tokens=False):
    """CLS vector (= the classifier's input) and, optionally, all token states, read with a hook on the encoder output."""
    cap = {}; hook = model.encoder.register_forward_hook(lambda m, i, o: cap.__setitem__("tok", o.last_hidden_state))
    model.eval(); cls_, toks = [], []
    with torch.no_grad():
        for b in batches(data, 64, False):
            with torch.autocast(DEVICE, dtype=torch.float16, enabled=DEVICE == "cuda"):
                model(**model_inputs(b, False))
            cls_.append(cap["tok"][:, 0].float().cpu())
            if want_tokens:
                toks.append(cap["tok"].half().cpu())
    hook.remove()
    return torch.cat(cls_), (torch.cat(toks) if want_tokens else None)

def text_run(rid, seed):
    """Train (or load) one text model; save checkpoint (fp16), log, probabilities and CLS features."""
    mode, use_eccm, loss_type = TEXT_RUNS[rid]; tag = f"{OUT_DIR}/text/{rid}_seed{seed}"
    if os.path.exists(tag + "_log.json"):
        log = json.load(open(tag + "_log.json")); assert log["config"] == CONFIG, f"{tag} was trained with other settings"
        return log, True
    t1 = time.time()
    model = train_run(rid, mode, use_eccm, loss_type, False, seed)
    secs = time.time() - t1
    out = {}
    for sp in ["valid", "test"]:
        out[f"probs_{sp}"] = predict(model, encoded[mode][sp], False).astype(np.float32)
    for sp in ["train", "valid", "test"]:
        out[f"cls_{sp}"] = text_states(model, encoded[mode][sp])[0].numpy().astype(np.float16)
    np.savez(tag + "_out.tmp.npz", **out); os.replace(tag + "_out.tmp.npz", tag + "_out.npz")
    torch.save({k: v.half() if v.is_floating_point() else v for k, v in model.state_dict().items()}, tag + ".tmp.pt"); os.replace(tag + ".tmp.pt", tag + ".pt")
    log = {"run": rid, "seed": seed, "mode": mode, "eccm": use_eccm, "loss": loss_type, "train_seconds": round(secs, 1), "config": CONFIG,
           "valid": evaluate(out["probs_valid"], encoded[mode]["valid"]), "test": evaluate(out["probs_test"], encoded[mode]["test"]),
           "parameters": sum(p.numel() for p in model.parameters())}
    json.dump(log, open(tag + "_log.tmp.json", "w"), indent=2, default=float); os.replace(tag + "_log.tmp.json", tag + "_log.json")
    del model; gc.collect(); torch.cuda.empty_cache() if DEVICE == "cuda" else None
    return log, False

jobs = [(r, s) for s in SEEDS for r in TEXT_RUNS]; t0 = time.time(); secs = []
for k, (rid, s) in enumerate(jobs, 1):
    log, loaded = text_run(rid, s)
    if not loaded:
        secs.append(log["train_seconds"])
    left = sum(not os.path.exists(f"{OUT_DIR}/text/{r}_seed{ss}_log.json") for r, ss in jobs[k:])
    print(f"[{k}/{len(jobs)}] {rid} seed {s}: {'loaded' if loaded else 'trained in %.0fs' % log['train_seconds']} | valid macro-F1 "
          f"{log['valid']['macro_f1']:.4f} | test macro-F1 {log['test']['macro_f1']:.4f} | elapsed {time.time() - t0:.0f}s"
          + (f" | about {np.mean(secs) * left / 60:.0f} min left for text runs" if secs else ""), flush=True)

  [R1] epoch 1/2  valid macro-F1 = 0.7249  (34s)
  [R1] epoch 2/2  valid macro-F1 = 0.7378  (69s)
[1/12] R1 seed 42: trained in 74s | valid macro-F1 0.7378 | test macro-F1 0.7334 | elapsed 88s | about 13 min left for text runs
  [R2] epoch 1/2  valid macro-F1 = 0.7988  (34s)
  [R2] epoch 2/2  valid macro-F1 = 0.8019  (69s)
[2/12] R2 seed 42: trained in 70s | valid macro-F1 0.8019 | test macro-F1 0.8174 | elapsed 173s | about 12 min left for text runs
  [R3] epoch 1/2  valid macro-F1 = 0.8028  (35s)
  [R3] epoch 2/2  valid macro-F1 = 0.8031  (70s)
[3/12] R3 seed 42: trained in 71s | valid macro-F1 0.8031 | test macro-F1 0.8190 | elapsed 258s | about 11 min left for text runs
  [R4] epoch 1/2  valid macro-F1 = 0.7953  (35s)
  [R4] epoch 2/2  valid macro-F1 = 0.8020  (71s)
[4/12] R4 seed 42: trained in 72s | valid macro-F1 0.8020 | test macro-F1 0.8148 | elapsed 345s | about 10 min left for text runs
  [R1] epoch 1/2  valid macro-F1 = 0.6960  (34s)
  [R1] epoch 2/2  valid macro-F1 = 0.744

## 6. Choose the text backbone for the heads — validation only (rule fixed above)

In [7]:
TLOG = {(r, s): json.load(open(f"{OUT_DIR}/text/{r}_seed{s}_log.json")) for r in TEXT_RUNS for s in SEEDS}
vt = pd.DataFrame([{"run": r, "seed": s, "valid_macro_f1": TLOG[(r, s)]["valid"]["macro_f1"],
                    "valid_worst_group_f1": TLOG[(r, s)]["valid"]["worst_group_f1"]} for r, s in TLOG]).groupby("run")[["valid_macro_f1", "valid_worst_group_f1"]].mean()
print((100 * vt).round(2).to_string())
best = vt.valid_macro_f1.max(); tied = vt[vt.valid_macro_f1 >= best - 0.005]
best_wg = tied.valid_worst_group_f1.max(); tied2 = tied[tied.valid_worst_group_f1 >= best_wg - 0.005]
SELECTED = [r for r in SIMPLICITY if r in tied2.index][0]
print(f"Tied on macro-F1 (within 0.5 pp of {100 * best:.2f}): {list(tied.index)} | then tied on worst-category F1: {list(tied2.index)}")
print(f"==> Text backbone for all heads: {SELECTED} ({TEXT_RUNS[SELECTED]}) — chosen on validation only.")
SEL_MODE, _, SEL_LOSS = TEXT_RUNS[SELECTED]
json.dump({"selected": SELECTED, "validation_means": vt.to_dict(), "rule": "max valid macro-F1; 0.5 pp ties -> max valid worst-group F1; "
           "0.5 pp ties -> simplest (R2<R1<R3<R4)"}, open(f"{OUT_DIR}/text_backbone_selection.json", "w"), indent=2)

     valid_macro_f1  valid_worst_group_f1
run                                      
R1            74.18                 72.02
R2            79.84                 78.15
R3            80.47                 79.22
R4            80.23                 78.51
Tied on macro-F1 (within 0.5 pp of 80.47): ['R3', 'R4'] | then tied on worst-category F1: ['R3']
==> Text backbone for all heads: R3 (('eccm', True, 'ce')) — chosen on validation only.


## 7. Fusion and baseline heads on the frozen features
All heads: AdamW (lr 1e-3, weight decay 1e-4), batch 64, up to 30 epochs, early stopping on validation macro-F1 (all photos present,
patience 5), the **same loss as the selected text model** (cross-entropy or Group-DRO), gradient clipping 1.0. Not tuned.

In [8]:
def masked_mean(x, mask):
    m = mask.unsqueeze(-1).float()
    return (x * m).sum(1) / m.sum(1).clamp(min=1)

class TextHead(nn.Module):                      # T0: same form as the text model's own classifier
    uses_tokens, uses_image = False, False
    def __init__(self):
        super().__init__(); self.cls = nn.Sequential(nn.Dropout(0.1), nn.Linear(768, 3))
    def forward(self, b):
        return self.cls(b["t_cls"])

class ImageHead(nn.Module):                     # IMG: lightweight image-only baseline
    uses_tokens, uses_image = False, True
    def __init__(self):
        super().__init__(); self.net = nn.Sequential(nn.Linear(768, D_HEAD), nn.ReLU(), nn.Dropout(0.1), nn.Linear(D_HEAD, 3))
    def forward(self, b):
        return self.net(b["i_mean"] * b["has"][:, None])

class ConcatHead(nn.Module):                    # CAT: simple concatenation
    uses_tokens, uses_image = False, True
    def __init__(self):
        super().__init__(); self.net = nn.Sequential(nn.Linear(1536, D_HEAD), nn.ReLU(), nn.Dropout(0.1), nn.Linear(D_HEAD, 3))
    def forward(self, b):
        return self.net(torch.cat([b["t_cls"], b["i_mean"] * b["has"][:, None]], -1))

class RMSR2(nn.Module):                         # F1 / SHUF: our adaptation of the paper's Eq. 10-14 to two modalities (as M2)
    uses_tokens, uses_image = False, True
    def __init__(self):
        super().__init__()
        self.proj = nn.ModuleDict({"text": nn.Linear(768, D_HEAD), "image": nn.Linear(768, D_HEAD)})               # Eq. 10
        self.u = nn.ParameterDict({m: nn.Parameter(torch.randn(D_HEAD) * 0.05) for m in ["text", "image"]})       # Eq. 11
        self.Wq, self.Wk, self.Wv = (nn.Linear(D_HEAD, D_HEAD) for _ in range(3))                                  # Eq. 13
        self.cls = nn.Sequential(nn.Dropout(0.2), nn.Linear(D_HEAD, 3))
    def forward(self, b, return_gate=False):
        z = {"text": self.proj["text"](b["t_cls"]), "image": self.proj["image"](b["i_mean"] * b["has"][:, None])}
        q = torch.stack([torch.sigmoid((z[m] * self.u[m]).sum(-1)) for m in ["text", "image"]], -1)               # Eq. 11
        gamma = q.softmax(-1)                                                                                       # Eq. 12
        score = (self.Wq(z["text"]) * self.Wk(z["image"])).sum(-1, keepdim=True) / D_HEAD ** 0.5                 # Eq. 13 (one key:
        att = score.softmax(-1) * self.Wv(z["image"])                                                              #  softmax = 1)
        h = gamma[:, :1] * z["text"] + gamma[:, 1:] * att                                                          # Eq. 14
        out = self.cls(h)
        return (out, gamma) if return_gate else out

class GatedFusion(nn.Module):                   # F2 (token attention, Notebook 1's design) / F3 (pooled, B3-style)
    def __init__(self, attention="token"):
        super().__init__()
        self.attention = attention; self.uses_tokens = attention == "token"; self.uses_image = True
        self.t_proj = nn.Sequential(nn.Linear(768, D_HEAD), nn.LayerNorm(D_HEAD))
        self.i_proj = nn.Sequential(nn.Linear(768, D_HEAD), nn.LayerNorm(D_HEAD))
        if attention == "token":
            self.xattn = nn.MultiheadAttention(D_HEAD, 4, batch_first=True, dropout=0.1)
        else:
            self.wv = nn.Linear(D_HEAD, D_HEAD)
        self.gate = nn.Linear(2 * D_HEAD, 2)
        self.cls = nn.Sequential(nn.Dropout(0.1), nn.Linear(D_HEAD, 3))
    def forward(self, b, return_gate=False):
        has = b["has"]
        p_i = self.i_proj(b["i_tok"]) * has[:, None, None]
        if self.attention == "token":
            p_t = self.t_proj(b["t_tok"]); h_t = p_t[:, 0]
            att, _ = self.xattn(p_t, p_i, p_i)                                   # every text token looks at the 50 photo tokens
            h_i = masked_mean(att, b["t_mask"])
        else:
            h_t = self.t_proj(b["t_cls"]); pooled = p_i.mean(1)
            h_i = self.wv(pooled) + pooled
        logits = self.gate(torch.cat([h_t, h_i], -1))
        logits = logits.masked_fill(torch.stack([torch.zeros_like(has, dtype=torch.bool), has == 0], -1), float("-inf"))
        w = logits.softmax(-1)
        out = self.cls(w[:, :1] * h_t + w[:, 1:] * h_i)
        return (out, w) if return_gate else out

HEADS = {"T0": TextHead, "IMG": ImageHead, "CAT": ConcatHead, "F1": RMSR2, "F2": lambda: GatedFusion("token"),
         "F3": lambda: GatedFusion("pooled"), "SHUF": RMSR2}
HEAD_ORDER = ["T0", "IMG", "CAT", "F1", "SHUF", "F3", "F2"]
USES_IMAGE = {"T0": False, "IMG": True, "CAT": True, "F1": True, "SHUF": True, "F2": True, "F3": True}

idx_of = {sp: np.where(df.split.values == sp)[0] for sp in ["train", "valid", "test"]}
assert all((encoded[SEL_MODE][sp]["row"].numpy() == idx_of[sp]).all() for sp in idx_of), "row order mismatch"
SHUF_PERM = {}                                   # fixed within-split permutation of photos for the SHUF control
for sp in idx_of:
    SHUF_PERM[sp] = idx_of[sp][np.random.RandomState(2026 + len(sp)).permutation(len(idx_of[sp]))]
MASK_FILE = f"{OUT_DIR}/missing_photo_masks.npz"  # paired hidden-photo masks, identical for every model
if not os.path.exists(MASK_FILE):
    n_te = len(idx_of["test"])
    np.savez(MASK_FILE, **{f"r{int(100 * r)}_m{m}": np.random.RandomState(1000 + 10 * m + int(100 * r)).rand(n_te) < r
                           for r in MISS_RATES for m in MASK_SEEDS})
MASKS = dict(np.load(MASK_FILE)); MASKS["r100_m0"] = np.ones(len(idx_of["test"]), dtype=bool)
EVAL_CONDS = ["r0"] + [k for k in MASKS]

def head_feats(seed, need_tokens):
    """Text features for every head are recomputed from the saved (fp16) text checkpoint, so all heads see identical features.
    They are compared with the CLS vectors saved right after training (full-precision weights) and the difference is printed."""
    z = np.load(f"{OUT_DIR}/text/{SELECTED}_seed{seed}_out.npz")
    feats, toks, cls = {}, {}, {}
    mode, use_eccm, _ = TEXT_RUNS[SELECTED]
    m = EmoFarModel(use_eccm=use_eccm, use_image=False)
    m.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in
                       torch.load(f"{OUT_DIR}/text/{SELECTED}_seed{seed}.pt", map_location="cpu").items()}); m.to(DEVICE)
    for sp in idx_of:
        cls[sp], toks[sp] = text_states(m, encoded[mode][sp], want_tokens=need_tokens)
    ref = z["cls_test"].astype(np.float32)
    rel = float(np.abs(cls["test"].numpy() - ref).max() / np.abs(ref).max())
    print(f"  seed {seed}: text features recomputed from the checkpoint; largest difference vs the saved ones = {100 * rel:.2f}% of the largest value")
    del m; torch.cuda.empty_cache() if DEVICE == "cuda" else None
    for sp in idx_of:
        rows = idx_of[sp]
        feats[sp] = {"t_cls": cls[sp].to(DEVICE),
                     "i_tok": torch.from_numpy(IMG_FEATS[rows]).to(DEVICE), "has": torch.from_numpy(HAS_IMG[rows].astype(np.float32)).to(DEVICE),
                     "t_mask": encoded[SEL_MODE][sp]["attention_mask"].to(DEVICE).bool(),
                     "y": encoded[SEL_MODE][sp]["label"].to(DEVICE), "g": encoded[SEL_MODE][sp]["group"].to(DEVICE)}
        feats[sp]["i_mean"] = feats[sp]["i_tok"].float().mean(1)
        rs = SHUF_PERM[sp]
        feats[sp]["i_tok_shuf"] = torch.from_numpy(IMG_FEATS[rs]).to(DEVICE)
        feats[sp]["i_mean_shuf"] = feats[sp]["i_tok_shuf"].float().mean(1)
        feats[sp]["has_shuf"] = torch.from_numpy(HAS_IMG[rs].astype(np.float32)).to(DEVICE)
        if need_tokens:
            feats[sp]["t_tok"] = toks[sp].to(DEVICE)
    return feats

def view(f, idx, name, has_override=None):
    """Batch view; SHUF uses the shuffled photos; has_override applies modality dropout or a hidden-photo mask."""
    shuf = name == "SHUF"
    b = {"t_cls": f["t_cls"][idx], "t_mask": f["t_mask"][idx], "i_mean": (f["i_mean_shuf"] if shuf else f["i_mean"])[idx],
         "has": (f["has_shuf"] if shuf else f["has"])[idx]}
    if "t_tok" in f:
        b["t_tok"] = f["t_tok"][idx].float()
    if name in ("F2", "F3"):
        b["i_tok"] = (f["i_tok_shuf"] if shuf else f["i_tok"])[idx].float()
    if has_override is not None:
        b["has"] = b["has"] * has_override
    return b

@torch.no_grad()
def head_probs(model, f, name, hide=None, bs=256):
    model.eval(); out = []; n = len(f["y"])
    for i in range(0, n, bs):
        idx = torch.arange(i, min(i + bs, n), device=DEVICE)
        ov = None if hide is None else torch.from_numpy((~hide[i:i + bs]).astype(np.float32)).to(DEVICE)
        out.append(model(view(f, idx, name, ov)).float().softmax(-1).cpu().numpy())
    return np.concatenate(out)

def head_loss(logits, y, g, q, loss_type):
    per = F.cross_entropy(logits, y, reduction="none")
    if loss_type != "gdro":
        return per.mean(), q
    gl = torch.stack([per[g == k].mean() if (g == k).any() else per.new_zeros(()) for k in range(len(CATEGORIES))])
    q = q * torch.exp(GDRO_ETA * gl.detach()); q = q / q.sum()
    return (q * gl).sum(), q

def train_head(name, seed, f):
    torch.manual_seed(seed); np.random.seed(seed)
    model = HEADS[name]().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=HEAD_LR, weight_decay=1e-4)
    rng = np.random.RandomState(seed); q = torch.ones(len(CATEGORIES), device=DEVICE) / len(CATEGORIES)
    tr, n = f["train"], len(f["train"]["y"]); drop = IMG_DROPOUT if name not in ("T0", "IMG") else 0.0
    best, best_state, bad, hist = -1, None, 0, []
    for ep in range(HEAD_EPOCHS):
        model.train(); order = rng.permutation(n)
        for i in range(0, n, HEAD_BS):
            idx = torch.from_numpy(order[i:i + HEAD_BS]).to(DEVICE)
            keep = torch.from_numpy((rng.rand(len(idx)) >= drop).astype(np.float32)).to(DEVICE)
            loss, q = head_loss(model(view(tr, idx, name, keep)), tr["y"][idx], tr["g"][idx], q, SEL_LOSS)
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        val = f1_score(f["valid"]["y"].cpu().numpy(), head_probs(model, f["valid"], name).argmax(1), average="macro")
        hist.append(float(val))
        if val > best:
            best, bad, best_state = val, 0, {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= HEAD_PATIENCE:
                break
    model.load_state_dict(best_state)
    return model, float(best), ep + 1, hist

t0 = time.time(); k = 0; n_jobs = len(SEEDS) * len(HEAD_ORDER)
for s in SEEDS:
    pending = [h for h in HEAD_ORDER if not os.path.exists(f"{OUT_DIR}/heads/{h}_seed{s}_log.json")]
    if not pending:
        k += len(HEAD_ORDER); print(f"seed {s}: all heads already done"); continue
    f = head_feats(s, need_tokens="F2" in pending)
    for h in HEAD_ORDER:
        k += 1; tag = f"{OUT_DIR}/heads/{h}_seed{s}"
        if os.path.exists(tag + "_log.json"):
            continue
        t1 = time.time(); model, best, ep, hist = train_head(h, s, f)
        probs = {"valid_r0": head_probs(model, f["valid"], h), "test_r0": head_probs(model, f["test"], h)}
        if USES_IMAGE[h]:
            for key, hide in MASKS.items():
                probs[f"test_{key}"] = head_probs(model, f["test"], h, hide)
        np.savez(tag + "_probs.tmp.npz", **probs); os.replace(tag + "_probs.tmp.npz", tag + "_probs.npz")
        torch.save(model.state_dict(), tag + ".pt")
        log = {"head": h, "seed": s, "text_backbone": SELECTED, "best_valid_macro_f1": best, "epochs": ep, "valid_history": hist,
               "train_seconds": round(time.time() - t1, 1), "parameters": sum(p.numel() for p in model.parameters()), "config": CONFIG}
        json.dump(log, open(tag + "_log.json", "w"), indent=2)
        print(f"  [{k}/{n_jobs}] {h} seed {s}: {log['train_seconds']:.0f}s, {ep} epochs, valid macro-F1 {best:.4f} | {time.time() - t0:.0f}s elapsed", flush=True)
        del model
    del f; gc.collect(); torch.cuda.empty_cache() if DEVICE == "cuda" else None

  seed 42: text features recomputed from the checkpoint; largest difference vs the saved ones = 0.11% of the largest value
  [1/21] T0 seed 42: 3s, 13 epochs, valid macro-F1 0.8087 | 17s elapsed
  [2/21] IMG seed 42: 2s, 7 epochs, valid macro-F1 0.4670 | 19s elapsed
  [3/21] CAT seed 42: 4s, 12 epochs, valid macro-F1 0.8073 | 22s elapsed
  [4/21] F1 seed 42: 3s, 6 epochs, valid macro-F1 0.8058 | 26s elapsed
  [5/21] SHUF seed 42: 4s, 8 epochs, valid macro-F1 0.8060 | 30s elapsed
  [6/21] F3 seed 42: 7s, 13 epochs, valid macro-F1 0.8088 | 37s elapsed
  [7/21] F2 seed 42: 12s, 13 epochs, valid macro-F1 0.8086 | 49s elapsed
  seed 43: text features recomputed from the checkpoint; largest difference vs the saved ones = 0.17% of the largest value
  [8/21] T0 seed 43: 2s, 9 epochs, valid macro-F1 0.8148 | 64s elapsed
  [9/21] IMG seed 43: 3s, 10 epochs, valid macro-F1 0.4603 | 67s elapsed
  [10/21] CAT seed 43: 2s, 7 epochs, valid macro-F1 0.8131 | 70s elapsed
  [11/21] F1 seed 43: 4s, 7 epo

## 7b. Pairing check without training: the trained F1 heads with test photos shuffled within each product category
Only reviews that have a photo exchange photos, and only with reviews of the same category, so photo availability and the category
of the photo are unchanged; each review gets a different review's photo (a shifted random order guarantees no review keeps its own).

In [9]:
N_SHUF_EVAL = 5
te_rows = idx_of["test"]; te_cat = df.category.values[te_rows]; te_has = HAS_IMG[te_rows]
SHUF_EVAL = []
for k in range(N_SHUF_EVAL):
    src = np.arange(len(te_rows)); rs = np.random.RandomState(5000 + k)
    for c in CATEGORIES:
        grp = np.where((te_cat == c) & te_has)[0]; order = grp[rs.permutation(len(grp))]
        src[order] = np.roll(order, 1)
    assert (te_has[src] == te_has).all() and (te_cat[src] == te_cat).all() and (src[te_has] != np.where(te_has)[0]).all()
    SHUF_EVAL.append(src)
for s in SEEDS:
    out_f = f"{OUT_DIR}/heads/F1_seed{s}_testshuf.npz"
    if os.path.exists(out_f):
        continue
    f = head_feats(s, need_tokens=False)
    m = RMSR2().to(DEVICE); m.load_state_dict(torch.load(f"{OUT_DIR}/heads/F1_seed{s}.pt", map_location=DEVICE)); m.eval()
    assert np.allclose(head_probs(m, f["test"], "F1"), np.load(f"{OUT_DIR}/heads/F1_seed{s}_probs.npz")["test_r0"], atol=1e-5), "F1 reload mismatch"
    res = {}
    for k, src in enumerate(SHUF_EVAL):
        st = torch.from_numpy(src).to(DEVICE); fp = dict(f["test"])
        fp["i_mean"], fp["i_tok"] = f["test"]["i_mean"][st], f["test"]["i_tok"][st]
        res[f"shuf{k}"] = head_probs(m, fp, "F1")
    np.savez(out_f, **res); print(f"  seed {s}: F1 evaluated with {N_SHUF_EVAL} within-category photo shuffles (no training)")
    del f, m; gc.collect(); torch.cuda.empty_cache() if DEVICE == "cuda" else None

  seed 42: text features recomputed from the checkpoint; largest difference vs the saved ones = 0.11% of the largest value
  seed 42: F1 evaluated with 5 within-category photo shuffles (no training)
  seed 43: text features recomputed from the checkpoint; largest difference vs the saved ones = 0.17% of the largest value
  seed 43: F1 evaluated with 5 within-category photo shuffles (no training)
  seed 44: text features recomputed from the checkpoint; largest difference vs the saved ones = 1.56% of the largest value
  seed 44: F1 evaluated with 5 within-category photo shuffles (no training)


## 8. Inference latency and parameter counts
**Protocol:** one GPU; the first 512 test reviews; **batch size 64 for every component** (text encoder, CLIP, heads); text encoder and
CLIP under fp16 autocast, heads in fp32; 2 warm-up passes, then the median of 10 timed passes, with CUDA synchronisation; reported as
ms per review. Included: moving the batch to the GPU and the forward pass (heads: also copying probabilities back to the CPU).
Excluded: photo download, image pre-processing (resize/normalise) and tokenisation. CLIP is timed on random 224×224 inputs
(its cost does not depend on the content). Total = text encoder + CLIP (if the model uses photos) + head.

In [10]:
def timed(fn, reps=10):
    for _ in range(2):
        fn()
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    ts = []
    for _ in range(reps):
        t1 = time.perf_counter(); fn()
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        ts.append(time.perf_counter() - t1)
    return float(np.median(ts))

LAT_FILE = f"{OUT_DIR}/latency_params_b64.csv"
LAT_PROTOCOL = {"batch_size_all_components": 64, "n_reviews": None, "warmup": 2, "repeats": 10, "statistic": "median",
                "precision": "fp16 autocast for text encoder and CLIP; fp32 heads", "excluded": "photo download, image pre-processing, tokenisation",
                "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"}
if not os.path.exists(LAT_FILE):
    NB = 512 if not TEST_MODE else 64
    mode, use_eccm, _ = TEXT_RUNS[SELECTED]
    tm = EmoFarModel(use_eccm=use_eccm, use_image=False)
    tm.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in torch.load(f"{OUT_DIR}/text/{SELECTED}_seed42.pt", map_location="cpu").items()})
    tm.to(DEVICE).eval(); te = {kk: v[:NB] for kk, v in encoded[mode]["test"].items()}
    def run_text():
        with torch.no_grad():
            for b in batches(te, 64, False):
                with torch.autocast(DEVICE, dtype=torch.float16, enabled=DEVICE == "cuda"):
                    tm(**model_inputs(b, False))
    t_text = timed(run_text) / NB * 1000
    clipm = CLIPVisionModel.from_pretrained(CLIP_NAME).to(DEVICE).eval(); px = torch.randn(64, 3, 224, 224, device=DEVICE)
    def run_clip():
        with torch.no_grad(), torch.autocast(DEVICE, dtype=torch.float16, enabled=DEVICE == "cuda"):
            for _ in range(NB // 64):
                clipm(pixel_values=px)
    t_clip = timed(run_clip) / NB * 1000
    p_text, p_clip = sum(p.numel() for p in tm.parameters()), sum(p.numel() for p in clipm.parameters())
    del tm, clipm; torch.cuda.empty_cache() if DEVICE == "cuda" else None
    f = head_feats(42, need_tokens=True); fte = {kk: v[:NB] for kk, v in f["test"].items()}; rows = []
    for h in HEAD_ORDER:
        m = HEADS[h]().to(DEVICE); m.load_state_dict(torch.load(f"{OUT_DIR}/heads/{h}_seed42.pt", map_location=DEVICE)); m.eval()
        t_head = timed(lambda: head_probs(m, fte, h, bs=64)) / NB * 1000
        uses_img = USES_IMAGE[h]; p_head = sum(p.numel() for p in m.parameters())
        rows.append({"model": h, "head_params": p_head, "text_encoder_params": p_text, "clip_params": p_clip if uses_img else 0,
                     "total_params_at_inference": p_text + p_head + (p_clip if uses_img else 0),
                     "ms_per_review_head": t_head, "ms_per_review_text_encoder": t_text, "ms_per_review_clip": t_clip if uses_img else 0.0,
                     "ms_per_review_total": t_text + t_head + (t_clip if uses_img else 0.0)})
    for r in [x for x in rows if x["model"] == "IMG"]:   # IMG does not need the text encoder at inference
        r = next(x for x in rows if x["model"] == "IMG"); r["text_encoder_params"] = 0; r["ms_per_review_text_encoder"] = 0.0
        r["total_params_at_inference"] = r["head_params"] + r["clip_params"]; r["ms_per_review_total"] = r["ms_per_review_head"] + r["ms_per_review_clip"]
    del f; gc.collect(); torch.cuda.empty_cache() if DEVICE == "cuda" else None
    pd.DataFrame(rows).to_csv(LAT_FILE, index=False)
    LAT_PROTOCOL["n_reviews"] = NB; json.dump(LAT_PROTOCOL, open(f"{OUT_DIR}/latency_protocol.json", "w"), indent=2)
LAT = pd.read_csv(LAT_FILE)
print("Latency protocol:", json.load(open(f"{OUT_DIR}/latency_protocol.json")))
print(LAT.round(4).to_string(index=False))

  seed 42: text features recomputed from the checkpoint; largest difference vs the saved ones = 0.11% of the largest value
Latency protocol: {'batch_size_all_components': 64, 'n_reviews': 512, 'warmup': 2, 'repeats': 10, 'statistic': 'median', 'precision': 'fp16 autocast for text encoder and CLIP; fp32 heads', 'excluded': 'photo download, image pre-processing, tokenisation', 'gpu': 'NVIDIA L4'}
model  head_params  text_encoder_params  clip_params  total_params_at_inference  ms_per_review_head  ms_per_review_text_encoder  ms_per_review_clip  ms_per_review_total
   T0         2307            124651782            0                  124654089              0.0047                      0.8645              0.0000               0.8692
  IMG       197635                    0     87456000                   87653635              0.0061                      0.0000              0.3705               0.3766
  CAT       394243            124651782     87456000                  212502025              0.

## 9. Statistics: Holm within the text family (4 tests) and the fusion family (6 tests); secondary missing-photo tests

In [11]:
test_df = df.iloc[idx_of["test"]]
y_te = test_df.label.values; g_te = np.array([CATEGORIES.index(c) for c in test_df.category])
has_neg_te = np.array([any(fl[0] for fl in e[1]) for e in test_df.eccm])
_, pidx = np.unique(test_df.parent_asin.values, return_inverse=True); G = pidx.max() + 1
_r = np.random.RandomState(2026)
W = np.stack([np.bincount(_r.randint(0, G, G), minlength=G) for _ in range(N_BOOT)]).astype(np.float32)
SD = np.random.RandomState(99).randint(0, len(SEEDS), (N_BOOT, len(SEEDS)))

def f1_from_counts(cm):
    cm = cm.reshape(cm.shape[:-1] + (3, 3)); tp = np.diagonal(cm, axis1=-2, axis2=-1)
    fp = cm.sum(-2) - tp; fn = cm.sum(-1) - tp; den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1e-12), 0).mean(-1)

def boot_scores(pred):
    """Bootstrap draws of macro-F1 and worst-category macro-F1 (resampling products)."""
    out = {}
    cm_all = np.zeros((G, 9)); np.add.at(cm_all, (pidx, y_te * 3 + pred), 1); out["macro_f1"] = f1_from_counts(W @ cm_all)
    per_cat = []
    for kk in range(len(CATEGORIES)):
        sel = g_te == kk; cm = np.zeros((G, 9)); np.add.at(cm, (pidx[sel], y_te[sel] * 3 + pred[sel]), 1); per_cat.append(f1_from_counts(W @ cm))
    out["worst_group_f1"] = np.min(per_cat, 0)
    return out

def point(pred, metric):
    if metric == "macro_f1":
        return f1_score(y_te, pred, average="macro")
    return min(f1_score(y_te[g_te == kk], pred[g_te == kk], average="macro") for kk in range(len(CATEGORIES)))

def _p(b):
    return min(1.0, 2 * (min((b <= 0).sum(), (b >= 0).sum()) + 1) / (len(b) + 1))

def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); m = len(p); adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, (m - rank) * p[i])); adj[i] = run
    return adj

def compare(pa, pb, metric):
    per_seed = np.array([point(a, metric) - point(b, metric) for a, b in zip(pa, pb)])
    d = np.stack([boot_scores(a)[metric] - boot_scores(b)[metric] for a, b in zip(pa, pb)])
    sv = d[SD, np.arange(N_BOOT)[:, None]].mean(1)
    return {"effect_pp": 100 * per_seed.mean(), "per_seed_pp": ", ".join(f"{100 * x:+.2f}" for x in per_seed),
            "lo_pp": 100 * np.percentile(sv, 2.5), "hi_pp": 100 * np.percentile(sv, 97.5), "p": _p(sv)}

def verdict(r):
    if r["p_holm"] < ALPHA and r["effect_pp"] >= MARGIN_PP: return "HELPS"
    if r["p_holm"] < ALPHA and r["effect_pp"] <= -MARGIN_PP: return "HURTS"
    if r["p_holm"] < ALPHA: return "significant, below 1 pp"
    if r["lo_pp"] > -MARGIN_PP and r["hi_pp"] < MARGIN_PP: return "negligible (CI within ±1 pp)"
    return "no detectable effect"

TP = {r: [np.load(f"{OUT_DIR}/text/{r}_seed{s}_out.npz")["probs_test"].argmax(1) for s in SEEDS] for r in TEXT_RUNS}
HP = {(h, c): [np.load(f"{OUT_DIR}/heads/{h}_seed{s}_probs.npz")[f"test_{c}"] for s in SEEDS]
      for h in HEAD_ORDER for c in EVAL_CONDS if f"test_{c}" in np.load(f"{OUT_DIR}/heads/{h}_seed42_probs.npz")}

fam_t = [("T1", "Conventional cleaning vs raw text", "R2", "R1", "macro_f1"), ("T2", "ECCM vs raw text", "R3", "R2", "macro_f1"),
         ("T3", "Group-DRO: overall cost/benefit", "R4", "R3", "macro_f1"), ("T4", "Group-DRO: worst category", "R4", "R3", "worst_group_f1")]
fam_f = [("P1", "Photo helps (main model)", "F1", "T0"), ("P2", "Real review-photo pairing", "F1", "SHUF"), ("P3", "Concatenation vs text only", "CAT", "T0"),
         ("P4", "Token attention design vs paper's RMSR", "F2", "F1"), ("P5", "Lightweight pooled vs paper's RMSR", "F3", "F1"),
         ("P6", "Concatenation vs paper's RMSR", "CAT", "F1")]
rows = [{"family": "text", "id": i, "question": q, "with": a, "without": b, "metric": m, **compare(TP[a], TP[b], m)} for i, q, a, b, m in fam_t]
rows += [{"family": "fusion", "id": i, "question": q, "with": a, "without": b, "metric": "macro_f1",
          **compare([p.argmax(1) for p in HP[(a, "r0")]], [p.argmax(1) for p in HP[(b, "r0")]], "macro_f1")} for i, q, a, b in fam_f]
fus_img = [h for h in ["F1", "F2", "F3", "CAT", "SHUF"]]
rows += [{"family": "secondary", "id": f"M-{h}", "question": f"{h}: all photos hidden vs none hidden", "with": f"{h} (100% hidden)",
          "without": f"{h} (0%)", "metric": "macro_f1",
          **compare([p.argmax(1) for p in HP[(h, "r100_m0")]], [p.argmax(1) for p in HP[(h, "r0")]], "macro_f1")} for h in fus_img]
def compare_vs_mean(pa, pb_lists, metric):
    """pa: per-seed predictions; pb_lists: per seed, a list of predictions (the shuffles), averaged inside every bootstrap draw."""
    per_seed = np.array([point(a, metric) - np.mean([point(b, metric) for b in bl]) for a, bl in zip(pa, pb_lists)])
    d = np.stack([boot_scores(a)[metric] - np.mean([boot_scores(b)[metric] for b in bl], 0) for a, bl in zip(pa, pb_lists)])
    sv = d[SD, np.arange(N_BOOT)[:, None]].mean(1)
    return {"effect_pp": 100 * per_seed.mean(), "per_seed_pp": ", ".join(f"{100 * x:+.2f}" for x in per_seed),
            "lo_pp": 100 * np.percentile(sv, 2.5), "hi_pp": 100 * np.percentile(sv, 97.5), "p": _p(sv)}
SHUFP = [[v.argmax(1) for v in dict(np.load(f"{OUT_DIR}/heads/F1_seed{s}_testshuf.npz")).values()] for s in SEEDS]
rows.append({"family": "secondary", "id": "S-F1", "question": "F1: real test photos vs within-category shuffled test photos (no training)",
             "with": "F1 (real photos)", "without": "F1 (shuffled test photos)", "metric": "macro_f1",
             **compare_vs_mean([p.argmax(1) for p in HP[("F1", "r0")]], SHUFP, "macro_f1")})
PRIM = pd.DataFrame(rows)
PRIM["p_holm"] = np.nan
for fam in PRIM.family.unique():
    sel = PRIM.family == fam; PRIM.loc[sel, "p_holm"] = holm(PRIM.loc[sel, "p"])
PRIM["verdict"] = PRIM.apply(verdict, axis=1)
PRIM.to_csv(f"{OUT_DIR}/tests.csv", index=False)

## 10. Results

In [12]:
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 50)
def metrics_row(name, probs_list, kind):
    rs = []
    for s, pr in zip(SEEDS, probs_list):
        pred = pr.argmax(1); per = {c: f1_score(y_te[g_te == kk], pred[g_te == kk], average="macro") for kk, c in enumerate(CATEGORIES)}
        rs.append({"model": name, "kind": kind, "seed": s, "accuracy": accuracy_score(y_te, pred), "macro_f1": f1_score(y_te, pred, average="macro"),
                   "worst_group_f1": min(per.values()), "category_gap": max(per.values()) - min(per.values()), "ece": ece_score(pr, y_te),
                   "negation_f1": f1_score(y_te[has_neg_te], pred[has_neg_te], average="macro"), **{f"f1_{c}": v for c, v in per.items()}})
    return rs
ALLR = []
for r in TEXT_RUNS:
    ALLR += metrics_row(r, [np.load(f"{OUT_DIR}/text/{r}_seed{s}_out.npz")["probs_test"] for s in SEEDS], "text model")
for h in HEAD_ORDER:
    ALLR += metrics_row(h, HP[(h, "r0")], f"head on {SELECTED}")
ALLR = pd.DataFrame(ALLR); ALLR.to_csv(f"{OUT_DIR}/results_per_seed.csv", index=False)
cols = ["accuracy", "macro_f1", "worst_group_f1", "category_gap", "ece", "negation_f1"] + [f"f1_{c}" for c in CATEGORIES]
summ = ALLR.groupby(["model", "kind"], sort=False)[cols].agg(["mean", "std"])
show = pd.DataFrame({c: [f"{100 * summ.loc[i, (c, 'mean')]:.2f} ± {100 * summ.loc[i, (c, 'std')]:.2f}" if c != "ece" else
                         f"{summ.loc[i, (c, 'mean')]:.4f} ± {summ.loc[i, (c, 'std')]:.4f}" for i in summ.index] for c in cols}, index=[i[0] for i in summ.index])
lat = LAT.set_index("model")[["head_params", "total_params_at_inference", "ms_per_review_total"]].copy()
t_enc = float(LAT.ms_per_review_text_encoder.max())
for r in TEXT_RUNS:                                   # R1-R4 share the same encoder; latency measured once (selected backbone)
    lat.loc[r] = [0, TLOG[(r, 42)]["parameters"], t_enc]
show = show.join(lat.round(3), how="left")
summ.to_csv(f"{OUT_DIR}/results_summary.csv")
print(f"Test results, mean ± SD over 3 seeds (%; ECE as a fraction). Heads use the {SELECTED} text backbone. Latency: ms per review, GPU, batch 64.")
print(show.to_string())

miss = []
for h in fus_img + ["IMG"]:
    for c in EVAL_CONDS:
        if (h, c) in HP:
            rate = 0 if c == "r0" else int(c.split("_")[0][1:])
            miss += [{"model": h, "hidden_rate": rate, "cond": c, "seed": s, "macro_f1": f1_score(y_te, p.argmax(1), average="macro")} for s, p in zip(SEEDS, HP[(h, c)])]
MISS = pd.DataFrame(miss); MISS.to_csv(f"{OUT_DIR}/missing_photo_per_seed.csv", index=False)
print("\nMacro-F1 (%) when photos are hidden for a share of test reviews (same masks for every model; mean over seeds and mask seeds)")
print((100 * MISS.groupby(["model", "hidden_rate"])["macro_f1"].mean().unstack()).round(2).to_string())
print(f"Reference: T0 (text only) macro-F1 {100 * ALLR[ALLR.model == 'T0'].macro_f1.mean():.2f}")

show_t = PRIM.copy()
show_t["effect [95% CI]"] = show_t.apply(lambda r: f"{r.effect_pp:+.2f} [{r.lo_pp:+.2f}, {r.hi_pp:+.2f}]", axis=1)
show_t["Holm p"] = show_t.p_holm.map(lambda x: f"{x:.3f}")
print("\nTESTS (effect = with − without, test, pp; Holm within each family)")
print(show_t[["family", "id", "question", "metric", "effect [95% CI]", "per_seed_pp", "Holm p", "verdict"]].to_string(index=False))
img = ALLR[ALLR.model == "IMG"].macro_f1
print(f"\nIMG (photo only) macro-F1: {100 * img.mean():.2f} ± {100 * img.std():.2f} (balanced 3 classes: chance level about 33.3)")
fs = np.mean([[f1_score(y_te, p, average="macro") for p in bl] for bl in SHUFP])
print(f"F1 with real test photos {100 * ALLR[ALLR.model == 'F1'].macro_f1.mean():.2f} vs within-category shuffled test photos {100 * fs:.2f} (no training)")
print("Reminder: photo-containing reviews only; labels from star ratings; balanced classes; missing photos simulated; "
      "fairness groups = product categories (not demographic groups).")
json.dump({"selected_text_backbone": SELECTED, "tests": PRIM.to_dict(orient="records"),
           "rules": {"n_boot": N_BOOT, "margin_pp": MARGIN_PP, "alpha": ALPHA, "holm": "within family"}},
          open(f"{OUT_DIR}/VERDICTS.json", "w"), indent=2, default=str)
print(f"\nSaved to {OUT_DIR}: tests.csv, results_per_seed.csv, results_summary.csv, missing_photo_per_seed.csv, latency_params_b64.csv, latency_protocol.json, VERDICTS.json")

Test results, mean ± SD over 3 seeds (%; ECE as a fraction). Heads use the R3 text backbone. Latency: ms per review, GPU, batch 64.
          accuracy      macro_f1 worst_group_f1 category_gap              ece   negation_f1 f1_Amazon_Fashion f1_All_Beauty  head_params  total_params_at_inference  ms_per_review_total
R1    74.12 ± 0.52  73.90 ± 0.53   72.78 ± 0.30  2.03 ± 0.73  0.0597 ± 0.0061  70.74 ± 0.61      72.78 ± 0.30  74.81 ± 0.84          0.0                124647939.0                0.865
R2    82.14 ± 0.22  82.03 ± 0.25   80.48 ± 0.48  2.92 ± 0.93  0.0522 ± 0.0039  80.49 ± 0.55      80.48 ± 0.48  83.40 ± 0.57          0.0                124647939.0                0.865
R3    82.01 ± 0.11  81.90 ± 0.03   80.29 ± 0.33  3.03 ± 0.69  0.0548 ± 0.0041  80.60 ± 0.59      80.29 ± 0.33  83.32 ± 0.36          0.0                124651782.0                0.865
R4    82.03 ± 0.47  81.91 ± 0.46   80.40 ± 0.57  2.86 ± 0.23  0.0522 ± 0.0086  80.69 ± 0.59      80.40 ± 0.57  83.26 ± 0.34     